# QUESTION 1 Total Marks [10]

This notebook **only computes locked Q1 numbers** from the frozen Drive splits (`train.parquet` / `val.parquet`). It does **not** train Logistic Regression, Random Forest, MLP, or the embedding net (Q3).

Cover: student names • student numbers • group number • lecturer • submission date — **TBD**.

Report prose: `/cursor/stores/self/docs/q1.md`.


## 1.1 Define the South African service-delivery problem, affected population, decision-maker and intended benefit. Convert it into a classification task by specifying the unit of prediction, inputs, target classes and prediction horizon. (4)

The problem, population, decision-maker, benefit, unit, inputs, classes and horizon are written in `q1.md`. Code below only **counts** the locked modelling table and checks that the **7-day** Fast/Slow rule is the target encoded in the Drive splits.

`Completed_Date` / `resolution_days` are **not** in the modelling parquets (leakage). The resolved-duration percentiles that justified the 7-day cut therefore stay as the locked team audit (among resolved requests: 28.4% same day, 56.2% within 2 days, **75.8% within 7 days**, 83.6% within 14, 89.4% within 30; median 2 days). This notebook verifies the **class mix** that that cut produced.


In [ ]:
from pathlib import Path
import importlib.util
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[misc,assignment]

CLASSES = ["Resolved Fast", "Resolved Slow", "Unresolved"]
SEED = 42


def find_data_dir() -> Path:
    """Frozen Drive splits: team-artifacts first, then repo data/processed."""
    candidates = [
        Path("/cursor/stores/self/internal/team-artifacts"),
        Path("/workspace/data/processed"),
        Path("data/processed"),
        Path("../data/processed"),
    ]
    for p in candidates:
        if (p / "train.parquet").exists():
            return p.resolve()
    raise FileNotFoundError(
        "Missing train.parquet. Put Drive splits in data/processed/ "
        "or keep them in /cursor/stores/self/internal/team-artifacts/."
    )


def load_prep(data_dir: Path):
    prep_candidates = [
        data_dir / "prep.py",
        Path("/workspace/src/prep.py"),
        Path("src/prep.py"),
        Path("../src/prep.py"),
    ]
    for path in prep_candidates:
        if path.exists():
            spec = importlib.util.spec_from_file_location("c3_prep", path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError("Could not find prep.py next to the splits or in src/.")


def figure_dirs() -> list[Path]:
    dirs = []
    store = Path("/cursor/stores/self/media/q2")
    try:
        store.mkdir(parents=True, exist_ok=True)
        dirs.append(store)
    except OSError:
        pass
    repo = Path("figures")
    if not repo.exists():
        alt = Path("../figures")
        repo = alt if alt.parent.exists() else repo
    repo.mkdir(parents=True, exist_ok=True)
    dirs.append(repo.resolve())
    # unique
    out, seen = [], set()
    for d in dirs:
        if d not in seen:
            out.append(d)
            seen.add(d)
    return out


DATA = find_data_dir()
prep = load_prep(DATA)
print("data_dir:", DATA)
print("prep module:", prep.__file__)
print("SEED", prep.SEED, "FEATURES", prep.FEATURES)
print("BANNED ∩ FEATURES", prep.BANNED & set(prep.FEATURES))


In [ ]:
train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")  # counts only — no EDA, no fit

def mix(df, name):
    n = len(df)
    vc = df["target"].astype(str).value_counts().reindex(CLASSES)
    out = pd.DataFrame({
        "set": name,
        "n": n,
        "Fast": int(vc["Resolved Fast"]),
        "Slow": int(vc["Resolved Slow"]),
        "Unresolved": int(vc["Unresolved"]),
        "Fast_%": 100 * vc["Resolved Fast"] / n,
        "Slow_%": 100 * vc["Resolved Slow"] / n,
        "Unresolved_%": 100 * vc["Unresolved"] / n,
    }, index=[0])
    return out

class_mix = pd.concat(
    [mix(train, "train"), mix(val, "val"), mix(test, "test"),
     mix(pd.concat([train, val, test], ignore_index=True), "modelling_total")],
    ignore_index=True,
)
class_mix


### 7-day cut from resolved durations

Locked audit (team Sections 3–5): among **resolved** requests, **75.8% finished within 7 whole SAST days**. That is the Fast/Slow cut (empirical ~p75). Median 2 days was rejected as harder to explain.

The modelling table has already applied that rule plus young-open (7 days) and 90-day maturity (created after 20 May 2026 dropped). The next cell checks: no duration columns leaked in; `target` is exactly the three classes.


In [ ]:
print("columns:", list(train.columns))
leaky = {"Completed_Date", "Changed_on", "resolution_days", "Notification",
         "Notification_type", "Suburb", "Work_Center"}
print("leaky columns present:", sorted(leaky & set(train.columns)))
print("target classes:", sorted(train["target"].astype(str).unique()))
print("created (train):", train["Created_On_Date"].min(), "→", train["Created_On_Date"].max())
print("7-day cut is encoded as target, not as a duration feature.")
print(class_mix[["set", "n", "Fast_%", "Slow_%", "Unresolved_%"]].round(2).to_string(index=False))


## 1.2 State one primary technical metric and at least one service-delivery success indicator. Explain the cost of false positives and false negatives. (2)

Primary technical metric: **macro-F1**. Service-delivery indicator and FP/FN costs: see `q1.md`.

Code: naïve `DummyClassifier` on the **frozen val fold** (seed 42). This is **not** one of the four substantive models. Test is not scored here.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

rows = []
for strategy in ("most_frequent", "stratified"):
    pipe = Pipeline([
        ("prep", prep.build_preprocessor()),
        ("model", DummyClassifier(strategy=strategy, random_state=SEED)),
    ])
    pipe.fit(train[prep.FEATURES], train["target"].astype(str))
    pred = pipe.predict(val[prep.FEATURES])
    y = val["target"].astype(str)
    report = classification_report(y, pred, labels=CLASSES, zero_division=0, output_dict=True)
    rows.append({
        "strategy": strategy,
        "accuracy_val": accuracy_score(y, pred),
        "macro_f1_val": f1_score(y, pred, average="macro"),
        "recall_slow_val": report["Resolved Slow"]["recall"],
        "recall_unresolved_val": report["Unresolved"]["recall"],
    })
    print(strategy)
    print(classification_report(y, pred, labels=CLASSES, zero_division=0))

dummy_val = pd.DataFrame(rows)
dummy_val


Expected locked val floor: most-frequent **macro-F1 ≈ 0.268** (accuracy ≈ 0.674, Slow/Unresolved recall 0); stratified **macro-F1 ≈ 0.333**.

## 1.3 Justify the dataset’s relevance, provenance, licence, coverage and limitations. Identify POPIA, privacy, fairness and misuse risks and propose safeguards. (4)

No extra fit. Coverage counts are the class-mix table above (modelling n = 3,029,242 from the three frozen folds). Provenance, licence, POPIA, fairness and safeguards: `q1.md`.
